In [1]:
import os
import numpy as np
import nibabel as nib

In [2]:
def load_nii(path):
    nii = nib.load(path)
    return nii.get_fdata()

In [1]:
def clip_hu(volume, min_hu=-200, max_hu=250):
    return np.clip(volume, min_hu, max_hu)

def normalize(volume):
    mean = np.mean(volume)
    std = np.std(volume)
    return (volume - mean) / (std + 1e-8)

def clean_mask(mask):
    return mask.astype(np.uint8)

def remove_empty_slices(volume, mask):

    v_slices = []
    m_slices = []

    # volume shape = (D, H, W)
    for i in range(volume.shape[0]):
        if np.sum(mask[i]) > 0:
            v_slices.append(volume[i])
            m_slices.append(mask[i])

    return np.array(v_slices), np.array(m_slices)

def extract_3d_patches(volume, mask,
                       patch_size=(16, 128, 128),
                       stride=16):

    D, H, W = volume.shape
    d, h, w = patch_size

    vol_patches = []
    mask_patches = []

    for z in range(0, D - d + 1, stride):
        for y in range(0, H - h + 1, stride):
            for x in range(0, W - w + 1, stride):

                v_patch = volume[z:z+d, y:y+h, x:x+w]
                m_patch = mask[z:z+d, y:y+h, x:x+w]

                if np.sum(m_patch) > 0:
                    vol_patches.append(v_patch)
                    mask_patches.append(m_patch)

    return vol_patches, mask_patches

In [2]:
def preprocess_image(image_path, mask_path):

    image = load_nii(image_path)
    mask = load_nii(mask_path)

    image = np.transpose(image, (2, 0, 1))
    mask = np.transpose(mask, (2, 0, 1))

    image = clip_hu(image)
    image = normalize(image)
    mask = clean_mask(mask)

    image, mask = remove_empty_slices(image, mask)

    if image.ndim != 3 or image.shape[0] < 16:
        return [], []

    img_patches, mask_patches = extract_3d_patches(image, mask)

    return img_patches, mask_patches

In [3]:
def load_dataset(image_dir, mask_dir):

    image_files = sorted(os.listdir(image_dir))

    X = []
    Y = []

    for file in image_files:

        if not file.endswith(".nii") and not file.endswith(".nii.gz"):
            continue

        image_path = os.path.join(image_dir, file)

        mask_file = file.replace("volume", "segmentation")
        mask_path = os.path.join(mask_dir, mask_file)

        if not os.path.exists(mask_path):
            print("Mask not found for:", file)
            continue

        img_patches, mask_patches = preprocess_image(image_path, mask_path)

        X.extend(img_patches)
        Y.extend(mask_patches)

    return np.array(X), np.array(Y)


In [6]:
image_dir = r"C:\\MiniProject\\Mini Project\\LiTS_small\\images"
mask_dir = r"C:\\MiniProject\\Mini Project\\LiTS_small\\masks"

X, Y = load_dataset(image_dir, mask_dir)

print("\nFinal dataset shape:")
print("Images:", X.shape)
print("Masks :", Y.shape)

MemoryError: Unable to allocate 51.6 GiB for an array with shape (26403, 16, 128, 128) and data type float64

In [2]:
import os
import numpy as np
import nibabel as nib


# ===============================
# Load NII File
# ===============================

def load_nii(path):
    nii = nib.load(path)
    data = np.array(nii.dataobj, dtype=np.float32)
    return data
# ===============================
# Preprocessing
# ===============================

def clip_hu(volume, min_hu=-200, max_hu=250):
    return np.clip(volume, min_hu, max_hu)


def normalize(volume):

    volume = volume.astype(np.float32)

    mean = volume.mean()
    std = volume.std()

    volume -= mean
    volume /= (std + 1e-8)

    return volume

def clean_mask(mask):
    return mask.astype(np.uint8)   # multi-class safe


# ===============================
# Remove Empty Slices
# ===============================

def remove_empty_slices(volume, mask):

    v_slices = []
    m_slices = []

    for i in range(volume.shape[0]):
        if np.sum(mask[i]) > 0:
            v_slices.append(volume[i])
            m_slices.append(mask[i])

    return np.array(v_slices), np.array(m_slices)


# ===============================
# Full Volume Processing (NO PATCHES)
# ===============================

def preprocess_image(image_path, mask_path):

    image = load_nii(image_path)
    mask = load_nii(mask_path)

    # LiTS orientation fix
    image = np.transpose(image, (2, 0, 1))
    mask = np.transpose(mask, (2, 0, 1))

    image = clip_hu(image).astype(np.float32)
    image = normalize(image)
    mask = clean_mask(mask)

    image, mask = remove_empty_slices(image, mask)

    if image.ndim != 3:
        return None, None

    return image.astype(np.float32), mask.astype(np.uint8)


# ===============================
# Load Dataset (Volume-wise)
# ===============================   
def load_dataset(image_dir, mask_dir):

    image_files = sorted(os.listdir(image_dir))

    volumes = []
    masks = []

    for file in image_files:

        if not file.endswith(".nii") and not file.endswith(".nii.gz"):
            continue

        image_path = os.path.join(image_dir, file)

        mask_file = file.replace("volume", "segmentation")
        mask_path = os.path.join(mask_dir, mask_file)

        if not os.path.exists(mask_path):
            print("Mask not found:", file)
            continue

        image, mask = preprocess_image(image_path, mask_path)

        if image is None:
            continue

        volumes.append(image)
        masks.append(mask)

        print(f"Loaded: {file} | Shape: {image.shape}")

    return volumes, masks

# ===============================
# RUN
# ===============================

image_dir = r"C:\Users\Dell\Desktop\3d CNN\data\volumes"
mask_dir  = r"C:\Users\Dell\Desktop\3d CNN\data\segmentations"

volumes, masks = load_dataset(image_dir, mask_dir)

print("\nTotal volumes loaded:", len(volumes))

Loaded: volume-0.nii | Shape: (29, 512, 512)
Loaded: volume-1.nii | Shape: (29, 512, 512)
Loaded: volume-10.nii | Shape: (181, 512, 512)
Loaded: volume-100.nii | Shape: (276, 512, 512)
Loaded: volume-101.nii | Shape: (259, 512, 512)
Loaded: volume-102.nii | Shape: (266, 512, 512)
Loaded: volume-103.nii | Shape: (214, 512, 512)
Loaded: volume-104.nii | Shape: (194, 512, 512)
Loaded: volume-105.nii | Shape: (239, 512, 512)
Loaded: volume-106.nii | Shape: (168, 512, 512)
Loaded: volume-107.nii | Shape: (248, 512, 512)
Loaded: volume-108.nii | Shape: (205, 512, 512)
Loaded: volume-109.nii | Shape: (194, 512, 512)
Loaded: volume-11.nii | Shape: (167, 512, 512)
Loaded: volume-110.nii | Shape: (189, 512, 512)
Loaded: volume-111.nii | Shape: (233, 512, 512)
Loaded: volume-112.nii | Shape: (191, 512, 512)
Loaded: volume-113.nii | Shape: (170, 512, 512)
Loaded: volume-114.nii | Shape: (215, 512, 512)
Loaded: volume-115.nii | Shape: (186, 512, 512)
Loaded: volume-116.nii | Shape: (219, 512, 512)


In [3]:
from sklearn.model_selection import train_test_split

volume_names = sorted(os.listdir(image_dir))

train_vols, val_vols = train_test_split(
    volume_names,
    test_size=0.2,
    random_state=42
)

print("Train volumes:", train_vols)
print("Val volumes:", val_vols)

Train volumes: ['volume-51.nii', 'volume-23.nii', 'volume-0.nii', 'volume-109.nii', 'volume-19.nii', 'volume-27.nii', 'volume-12.nii', 'volume-41.nii', 'volume-83.nii', 'volume-61.nii', 'volume-111.nii', 'volume-50.nii', 'volume-118.nii', 'volume-20.nii', 'volume-4.nii', 'volume-68.nii', 'volume-125.nii', 'volume-37.nii', 'volume-94.nii', 'volume-106.nii', 'volume-128.nii', 'volume-40.nii', 'volume-120.nii', 'volume-88.nii', 'volume-123.nii', 'volume-7.nii', 'volume-98.nii', 'volume-29.nii', 'volume-102.nii', 'volume-42.nii', 'volume-47.nii', 'volume-16.nii', 'volume-25.nii', 'volume-13.nii', 'volume-112.nii', 'volume-95.nii', 'volume-129.nii', 'volume-58.nii', 'volume-104.nii', 'volume-2.nii', 'volume-5.nii', 'volume-75.nii', 'volume-80.nii', 'volume-105.nii', 'volume-11.nii', 'volume-66.nii', 'volume-100.nii', 'volume-113.nii', 'volume-15.nii', 'volume-46.nii', 'volume-72.nii', 'volume-103.nii', 'volume-56.nii', 'volume-82.nii', 'volume-84.nii', 'volume-3.nii', 'volume-26.nii', 'volu

In [4]:
def extract_3d_patches(volume, mask,
                       patch_size=(16,128,128),
                       stride=(8,64,64)):

    D, H, W = volume.shape
    d, h, w = patch_size
    sd, sh, sw = stride

    vol_patches = []
    mask_patches = []

    for z in range(0, D - d + 1, sd):
        for y in range(0, H - h + 1, sh):
            for x in range(0, W - w + 1, sw):

                v_patch = volume[z:z+d, y:y+h, x:x+w]
                m_patch = mask[z:z+d, y:y+h, x:x+w]

                if np.sum(m_patch == 2) > 0:   # tumor
                    vol_patches.append(v_patch)
                    mask_patches.append(m_patch)

                elif np.sum(m_patch == 1) > 0:  # liver
                    vol_patches.append(v_patch)
                    mask_patches.append(m_patch)

                elif np.random.rand() < 0.05:   # background
                    vol_patches.append(v_patch)
                    mask_patches.append(m_patch)

    return vol_patches, mask_patches

In [5]:
def create_patches_from_volumes(volume_list, image_dir, mask_dir):

    X = []
    Y = []

    for file in volume_list:

        image_path = os.path.join(image_dir, file)
        mask_file = file.replace("volume", "segmentation")
        mask_path = os.path.join(mask_dir, mask_file)

        image, mask = preprocess_image(image_path, mask_path)

        if image is None:
            continue

        img_patches, mask_patches = extract_3d_patches(image, mask)

        X.extend(img_patches)
        Y.extend(mask_patches)

        print(f"{file} → {len(img_patches)} patches")

    return np.array(X, dtype=np.float32), \
           np.array(Y, dtype=np.uint8)

In [6]:
print("\nCreating TRAIN patches...")
X_train, Y_train = create_patches_from_volumes(
    train_vols, image_dir, mask_dir
)

print("\nCreating VAL patches...")
X_val, Y_val = create_patches_from_volumes(
    val_vols, image_dir, mask_dir
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)


Creating TRAIN patches...
volume-51.nii → 130 patches
volume-23.nii → 298 patches
volume-0.nii → 53 patches
volume-109.nii → 503 patches
volume-19.nii → 506 patches
volume-27.nii → 571 patches
volume-12.nii → 465 patches
volume-41.nii → 281 patches
volume-83.nii → 524 patches
volume-61.nii → 160 patches
volume-111.nii → 572 patches
volume-50.nii → 162 patches
volume-118.nii → 337 patches
volume-20.nii → 521 patches
volume-4.nii → 671 patches
volume-68.nii → 187 patches
volume-125.nii → 306 patches
volume-37.nii → 266 patches
volume-94.nii → 568 patches
volume-106.nii → 414 patches
volume-128.nii → 775 patches
volume-40.nii → 256 patches
volume-120.nii → 308 patches
volume-88.nii → 443 patches
volume-123.nii → 364 patches
volume-7.nii → 390 patches
volume-98.nii → 640 patches
volume-29.nii → 279 patches
volume-102.nii → 632 patches
volume-42.nii → 262 patches
volume-47.nii → 207 patches
volume-16.nii → 509 patches
volume-25.nii → 582 patches
volume-13.nii → 324 patches
volume-112.nii →

In [7]:
np.save("X_train.npy", X_train)
np.save("Y_train.npy", Y_train)

np.save("X_val.npy", X_val)
np.save("Y_val.npy", Y_val)

print("Saved successfully.")

Saved successfully.
